# PyCisTopic exploration

Use this to install: https://github.com/aertslab/scenicplus/issues/586
and install this version setuptools==79.0.1

In [3]:
import os
import pickle 
os.environ['RUST_BACKTRACE'] = '1'

os.chdir("/projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic")


In [4]:
fragments_dict = {
    "Deepseq": "/projectnb/paxlab/presh/projects/spatial_atac/Data/deepseq.fragments.sort.filtered.bed.gz"
}

out_dir = "outs"
os.makedirs(out_dir, exist_ok = True)

In [5]:

# fragments_dict = {
#     "10x_multiome_brain": "data/fragments.tsv.gz"
# }

# import pandas as pd
# cell_data = pd.read_table("data/cell_data.tsv", index_col = 0)
# cell_data.head()

In [6]:
import pandas as pd
cell_data = pd.read_csv(
    "/projectnb/paxlab/presh/projects/spatial_atac/Data/archr_metadata.tsv",
    index_col=0,
    sep='\s+'
)

cell_data.index = cell_data.index.str.replace(r'^Deepseq#', '', regex=True) # .str.replace('-1$', '', regex=True)
cell_data.head()

,Sample,TSSEnrichment,ReadsInTSS,ReadsInPromoter,ReadsInBlacklist,PromoterRatio,PassQC,NucleosomeRatio,nMultiFrags,nMonoFrags,...,nDiFrags,DoubletScore,DoubletEnrichment,BlacklistRatio,tissue,Clusters_tile,Clusters_gene,ReadsInPeaks,FRIP,Subclone_cluster
ACTCTTGCAGTCCTTG-1,Deepseq,4.054,18976,27076,3376,0.067739,1,0.489695,0,134159,...,65697,0.0,0.688889,0.008446,Tiss_488B,C3,C4,73396,0.183652,6
AGTGATCGGCGAGTAA-1,Deepseq,4.030,18743,27932,2957,0.070170,1,0.431644,0,139022,...,60008,0.0,0.311111,0.007429,Tiss_488B,C1,C2,90615,0.227670,2
CTTCCTTGTTGATGCC-1,Deepseq,2.747,12099,20606,3205,0.051767,1,0.416109,0,140545,...,58482,0.0,0.822222,0.008052,Tiss_488B,C1,C2,69499,0.174617,3
GATAGTCGAGTCCTTG-1,Deepseq,4.423,19009,26667,3134,0.067092,1,0.555623,0,127752,...,70982,0.0,0.522222,0.007885,Tiss_488B,C3,C2,72026,0.181245,6
CTAACGTGTAATGCGG-1,Deepseq,3.431,15273,23699,3066,0.060132,1,0.412751,0,139486,...,57573,0.0,0.755556,0.007779,Tiss_488B,C2,C2,76249,0.193493,4


In [7]:
# Check unique samples in your cell metadata
unique_samples = cell_data["Sample"].unique()
print("Unique samples in cell_data:", unique_samples)
print("Number of unique samples:", len(unique_samples))

# Check keys in fragments_dict
print("\nKeys in fragments_dict:", list(fragments_dict.keys()))
print("Number of keys in fragments_dict:", len(fragments_dict.keys()))

# Find missing samples
missing_samples = set(unique_samples) - set(fragments_dict.keys())
print("\nMissing fragment paths for:", missing_samples)

Unique samples in cell_data: ['Deepseq']
Number of unique samples: 1

Keys in fragments_dict: ['Deepseq']
Number of keys in fragments_dict: 1

Missing fragment paths for: set()


In [8]:
chromsizes = pd.read_table(
    "http://hgdownload.cse.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes",
    header = None,
    names = ["Chromosome", "End"]
)
chromsizes.insert(1, "Start", 0)
chromsizes.head()

,Chromosome,Start,End
0,chr1,0,248956422
1,chr2,0,242193529
2,chr3,0,198295559
3,chr4,0,190214555
4,chr5,0,181538259


# Generate pseudobulk

In [ ]:
from pycisTopic.pseudobulk_peak_calling import export_pseudobulk
os.makedirs(os.path.join(out_dir, "consensus_peak_calling"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bed_files"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bw_files"), exist_ok = True)

temp_dir = os.path.join(out_dir, "temp")

bw_paths, bed_paths = export_pseudobulk(
    input_data = cell_data,
    variable = "Clusters_tile",
    sample_id_col = "Sample",
    chromsizes = chromsizes,
    bed_path = os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bed_files"),
    bigwig_path = os.path.join(out_dir, "consensus_peak_calling/pseudobulk_bw_files"),
    path_to_fragments = fragments_dict,
    n_cpu = 2,
    normalize_bigwig = True,
    temp_dir = temp_dir,
    split_pattern = "-"
)

In [ ]:

with open(os.path.join(out_dir, "consensus_peak_calling/bw_paths.tsv"), "wt") as f:
    for v in bw_paths:
        _ = f.write(f"{v}\t{bw_paths[v]}\n")

In [ ]:
with open(os.path.join(out_dir, "consensus_peak_calling/bed_paths.tsv"), "wt") as f:
    for v in bed_paths:
        _ = f.write(f"{v}\t{bed_paths[v]}\n")

# Inferring consensus peaks

In [ ]:
bw_paths = {}
with open(os.path.join(out_dir, "consensus_peak_calling/bw_paths.tsv")) as f:
    for line in f:
        v, p = line.strip().split("\t")
        bw_paths.update({v: p})

bed_paths = {}
with open(os.path.join(out_dir, "consensus_peak_calling/bed_paths.tsv")) as f:
    for line in f:
        v, p = line.strip().split("\t")
        bed_paths.update({v: p})

In [ ]:
from pycisTopic.pseudobulk_peak_calling import peak_calling
macs_path = "macs2"

os.makedirs(os.path.join(out_dir, "consensus_peak_calling/MACS"), exist_ok = True)

temp_dir = os.path.join(out_dir, "temp")

narrow_peak_dict = peak_calling(
    macs_path = macs_path,
    bed_paths = bed_paths,
    outdir = os.path.join(os.path.join(out_dir, "consensus_peak_calling/MACS")),
    genome_size = 'hs',
    n_cpu = 10,
    input_format = 'BEDPE',
    shift = 73,
    ext_size = 146,
    keep_dup = 'all',
    q_value = 0.05,
    _temp_dir = '/scratch/leuven/330/vsc33053/ray_spill'
)

In [ ]:
from pycisTopic.iterative_peak_calling import get_consensus_peaks
# Other param
peak_half_width=250
path_to_blacklist="/projectnb/paxlab/presh/software/pycisTopic/blacklist/hg38-blacklist.v2.bed"
# Get consensus peaks
consensus_peaks = get_consensus_peaks(
    narrow_peaks_dict = narrow_peak_dict,
    peak_half_width = peak_half_width,
    chromsizes = chromsizes,
    path_to_blacklist = path_to_blacklist)

In [ ]:
consensus_peaks.to_bed(
    path = os.path.join(out_dir, "consensus_peak_calling/consensus_regions.bed"),
    keep =True,
    compression = 'infer',
    chain = False)

# QC

In [ ]:
!pycistopic tss gene_annotation_list | grep Human

In [ ]:
# !mkdir -p outs/qc
# !pycistopic tss get_tss \
#     --output outs/qc/tss.bed \
#     --name "hsapiens_gene_ensembl" \
#     --to-chrom-source ucsc \
#     --ucsc hg38

In [ ]:
import gzip
import pandas as pd

genes = []

with gzip.open('outs/qc/Homo_sapiens.GRCh38.110.gtf.gz', 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        
        fields = line.strip().split('\t')
        if len(fields) < 9 or fields[2] != 'gene':
            continue
        
        attrs = {}
        for attr in fields[8].split(';'):
            attr = attr.strip()
            if not attr or ' ' not in attr:
                continue
            key, val = attr.split(' ', 1)
            attrs[key] = val.strip('"')
        
        if attrs.get('gene_biotype') != 'protein_coding':
            continue
        
        chrom = fields[0]
        
        # Convert to UCSC format
        if not chrom.startswith('chr'):
            if chrom == 'MT':
                chrom = 'chrM'
            else:
                chrom = f'chr{chrom}'
        
        strand = fields[6]
        start_pos = int(fields[3])
        end_pos = int(fields[4])
        tss = start_pos if strand == '+' else end_pos
        gene_name = attrs.get('gene_name', attrs.get('gene_id', 'unknown'))
        
        genes.append([chrom, tss - 1, tss, gene_name, '.', strand, 'protein_coding'])

tss_df = pd.DataFrame(genes, columns=['Chromosome', 'Start', 'End', 'Gene', 'Score', 'Strand', 'Transcript_type'])
tss_df = tss_df.drop_duplicates()

# Natural sort
def sort_chroms(chrom):
    if chrom == 'chrM':
        return (999, 0)
    elif chrom == 'chrX':
        return (23, 0)
    elif chrom == 'chrY':
        return (24, 0)
    else:
        try:
            num = int(chrom.replace('chr', ''))
            return (num, 0)
        except:
            return (1000, 0)

tss_df['chr_sort'] = tss_df['Chromosome'].apply(sort_chroms)
tss_df = tss_df.sort_values(['chr_sort', 'Start']).drop('chr_sort', axis=1)

# Save WITH # header
with open('outs/qc/tss.bed', 'w') as f:
    # Write header with #
    f.write("# Chromosome\tStart\tEnd\tGene\tScore\tStrand\tTranscript_type\n")
    # Write data
    tss_df.to_csv(f, sep='\t', index=False, header=False)

print(f"✓ Created {len(tss_df)} TSS annotations with # header")
print("\nFirst 10 lines:")
with open('outs/qc/tss.bed', 'r') as f:
    for i, line in enumerate(f):
        if i < 10:
            print(line.rstrip())
        else:
            break

In [ ]:
!head outs/qc/tss.bed | column -t

In [ ]:
!pycistopic qc \
    --fragments /projectnb/paxlab/presh/projects/spatial_atac/Data/deepseq.fragments.sort.filtered.bed.gz \
    --regions outs/consensus_peak_calling/consensus_regions.bed \
    --tss outs/qc/tss.bed \
    --output outs/qc/Deepseq

In [ ]:
# !pip install diptest

In [ ]:
from pycisTopic.plotting.qc_plot import plot_sample_stats, plot_barcode_stats
import matplotlib.pyplot as plt

In [ ]:
# for sample_id in fragments_dict:
#     fig = plot_sample_stats(
#         sample_id = sample_id,
#         pycistopic_qc_output_dir = "outs/qc"
#     )


In [ ]:
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt

# Now run your plot
for sample_id in fragments_dict:
    fig = plot_sample_stats(
        sample_id = sample_id,
        pycistopic_qc_output_dir = "outs/qc"
    )
    plt.savefig(f'outs/qc/{sample_id}_qc.png', dpi=300, bbox_inches='tight')
    plt.close(fig)

In [ ]:
from pycisTopic.qc import get_barcodes_passing_qc_for_sample
sample_id_to_barcodes_passing_filters = {}
sample_id_to_thresholds = {}
for sample_id in fragments_dict:
    (
        sample_id_to_barcodes_passing_filters[sample_id],
        sample_id_to_thresholds[sample_id]
    ) = get_barcodes_passing_qc_for_sample(
            sample_id = sample_id,
            pycistopic_qc_output_dir = "outs/qc",
            unique_fragments_threshold = None, # use automatic thresholding
            tss_enrichment_threshold = None, # use automatic thresholding
            frip_threshold = 0,
            use_automatic_thresholds = True,
    )

In [ ]:
# for sample_id in fragments_dict:
#     fig = plot_barcode_stats(
#         sample_id = sample_id,
#         pycistopic_qc_output_dir = "outs/qc",
#         bc_passing_filters = sample_id_to_barcodes_passing_filters[sample_id],
#         detailed_title = False,
#         **sample_id_to_thresholds[sample_id]
#     )

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pycisTopic.plotting.qc_plot import plot_barcode_stats

# Now plot
for sample_id in fragments_dict:
    fig = plot_barcode_stats(
        sample_id = sample_id,
        pycistopic_qc_output_dir = "outs/qc",
        bc_passing_filters = sample_id_to_barcodes_passing_filters[sample_id],
        detailed_title = False,
        **sample_id_to_thresholds[sample_id]
    )
    plt.savefig(f'outs/qc/{sample_id}_barcode_stats.png', dpi=300, bbox_inches='tight')
    plt.close(fig)

print(f"✓ Barcode stats plots saved for {len(fragments_dict)} samples")

# Create a cistopic obj 

In [ ]:
path_to_regions = os.path.join(out_dir, "consensus_peak_calling/consensus_regions.bed")
path_to_blacklist = "/projectnb/paxlab/presh/software/pycisTopic/blacklist/hg38-blacklist.v2.bed"
pycistopic_qc_output_dir = "outs/qc"

from pycisTopic.cistopic_class import create_cistopic_object_from_fragments
import polars as pl

cistopic_obj_list = []
for sample_id in fragments_dict:
    sample_metrics = pl.read_parquet(
        os.path.join(pycistopic_qc_output_dir, f'{sample_id}.fragments_stats_per_cb.parquet')
    ).to_pandas().set_index("CB").loc[ sample_id_to_barcodes_passing_filters[sample_id] ]
    cistopic_obj = create_cistopic_object_from_fragments(
        path_to_fragments = fragments_dict[sample_id],
        path_to_regions = path_to_regions,
        path_to_blacklist = path_to_blacklist,
        metrics = sample_metrics,
        valid_bc = sample_id_to_barcodes_passing_filters[sample_id],
        n_cpu = 8,
        project = sample_id,
        split_pattern = '-'
    )
    cistopic_obj_list.append(cistopic_obj)

In [ ]:
cistopic_obj = cistopic_obj_list[0]
print(cistopic_obj)

In [8]:
import pickle
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

NameError: name 'cistopic_obj' is not defined

# Adding metadata

In [15]:
import pandas as pd
cell_data = pd.read_table("/projectnb/paxlab/presh/projects/spatial_atac/Data/archr_metadata.tsv", index_col = 0, sep='\s+')
cell_data.index = cell_data.index.str.replace(r'^Deepseq#', '', regex=True)
cell_data.head()
cistopic_obj.add_cell_data(cell_data, split_pattern='-')
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

Columns ['Clusters_tile', 'ReadsInTSS', 'nMonoFrags', 'PromoterRatio', 'ReadsInPeaks', 'DoubletEnrichment', 'tissue', 'ReadsInPromoter', 'FRIP', 'DoubletScore', 'nMultiFrags', 'nFrags', 'Sample', 'nDiFrags', 'PassQC', 'NucleosomeRatio', 'BlacklistRatio', 'Clusters_gene', 'TSSEnrichment', 'Subclone_cluster', 'ReadsInBlacklist'] will be overwritten


In [19]:
print("Projections:", list(cistopic_obj.projections.get('cell', {}).keys()))
print("Selected model:", cistopic_obj.selected_model)

Projections: ['UMAP', 'tSNE']
Selected model: CistopicLDAModel with 30 topics and n_cells × n_regions = 2650 × 1029858


In [ ]:
print(cistopic_obj.cell_data.columns)

Index(['cisTopic_log_nr_frag', 'pycisTopic_leiden_10_1.2', 'barcode_rank',
       'pdf_values_for_tss_enrichment', 'Seurat_leiden_res0.6',
       'cisTopic_nr_frag', 'pdf_values_for_fraction_of_fragments_in_peaks',
       'barcode', 'Doublet_scores_fragments', 'tss_enrichment',
       'fraction_of_fragments_in_peaks', 'total_fragments_in_peaks_count',
       'log10_unique_fragments_in_peaks_count', 'total_fragments_count',
       'nucleosome_signal', 'cisTopic_log_nr_acc',
       'log10_unique_fragments_count', 'pdf_values_for_duplication_ratio',
       'VSN_sample_id', 'pycisTopic_leiden_10_0.6', 'VSN_leiden_res1.2',
       'log10_total_fragments_in_peaks_count', 'cisTopic_nr_acc',
       'duplication_count', 'Seurat_cell_type', 'VSN_leiden_res0.3',
       'Predicted_doublets_fragments', 'log10_total_fragments_count',
       'sample_id', 'unique_fragments_count', 'VSN_leiden_res0.9',
       'VSN_leiden_res0.6', 'duplication_ratio', 'VSN_cell_type',
       'unique_fragments_in_peaks_co

In [ ]:
cistopic_obj.cell_data.to_csv('cell_metadata.csv', sep=',')

# Running scrublet 

In [ ]:
import scrublet as scr

import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
# Load the object
cistopic_obj = pickle.load(
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "rb")
)

scrub = scr.Scrublet(cistopic_obj.fragment_matrix.T, expected_doublet_rate=0.1)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

# Save plot instead of showing
scrub.plot_histogram()
plt.savefig('outs/qc/scrublet_histogram.png', dpi=300, bbox_inches='tight')
plt.close()

scrub.call_doublets(threshold=0.22)
scrub.plot_histogram()
plt.savefig('outs/qc/scrublet_histogram_thresh.png', dpi=300, bbox_inches='tight')
plt.close()

scrublet = pd.DataFrame([scrub.doublet_scores_obs_, scrub.predicted_doublets_], columns=cistopic_obj.cell_names, index=['Doublet_scores_fragments', 'Predicted_doublets_fragments']).T

Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.66
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 0.2%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 88.9%
Elapsed time: 50.1 seconds
Detected doublet rate = 31.9%
Estimated detectable doublet fraction = 58.2%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 54.8%


In [ ]:
cistopic_obj.add_cell_data(scrublet, split_pattern = '-')
sum(cistopic_obj.cell_data.Predicted_doublets_fragments == True)

Columns ['Doublet_scores_fragments', 'Predicted_doublets_fragments'] will be overwritten


845

In [ ]:
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

In [ ]:
# Remove doublets
singlets = cistopic_obj.cell_data[cistopic_obj.cell_data.Predicted_doublets_fragments == False].index.tolist()
# Subset cisTopic object
cistopic_obj_noDBL = cistopic_obj.subset(singlets, copy=True, split_pattern='-')
print(cistopic_obj_noDBL)

CistopicObject from project Deepseq with n_cells × n_regions = 1840 × 1029858


In [ ]:
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

# Run models

In [ ]:
# !cd /projectnb/paxlab/presh/software
# !wget https://github.com/mimno/Mallet/releases/download/v202108/Mallet-202108-bin.tar.gz
# !tar -xf Mallet-202108-bin.tar.gz

In [ ]:
!mkdir -p /scratch/leuven/330/vsc33053/ray_spill/mallet/tutorial/

In [14]:
import pickle
import os

out_dir = "outs"  # or wherever you saved it

# Load the object
cistopic_obj = pickle.load(
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "rb")
)

print("✓ Loaded cistopic object")
print(f"Type: {type(cistopic_obj)}")
print(f"Object info: {cistopic_obj}")

✓ Loaded cistopic object
Type: <class 'pycisTopic.cistopic_class.CistopicObject'>
Object info: CistopicObject from project Deepseq with n_cells × n_regions = 2650 × 1029858


In [ ]:
os.environ['MALLET_MEMORY'] = '200G'
from pycisTopic.lda_models import run_cgs_models_mallet
# Configure path Mallet
mallet_path="/projectnb/paxlab/presh/software/Mallet-202108/bin/mallet"
# Run models
models=run_cgs_models_mallet(
    cistopic_obj,
    n_topics=[2, 5], #10, 15, 20, 25, 30, 35, 40, 45, 50
    n_cpu=12,
    n_iter=500,
    random_state=555,
    alpha=50,
    alpha_by_topic=True,
    eta=0.1,
    eta_by_topic=False,
    tmp_path="/scratch/leuven/330/vsc33053/ray_spill/mallet/tutorial",
    save_path="/scratch/leuven/330/vsc33053/ray_spill/mallet/tutorial",
    mallet_path=mallet_path,
)

In [ ]:
pickle.dump(
    models,
    open(os.path.join(out_dir, "models.pkl"), "wb")
)

# Model selection

I did 2 pickle objects separately, combining them below now

In [ ]:
import pickle
import os

out_dir = "outs"

# Load all pickle files
with open(os.path.join(out_dir, "lda_models.pkl"), 'rb') as f:
    models1 = pickle.load(f)

with open(os.path.join(out_dir, "lda_models2.pkl"), 'rb') as f:
    models2 = pickle.load(f)

# Combine into single flat list
all_models = list(models1) + list(models2)

print(f"Total models: {len(all_models)}")
print(f"Models 1: {len(models1)}, Models 2: {len(models2)}")

# Save combined list
with open(os.path.join(out_dir, "combined_lda_models.pkl"), 'wb') as f:
    pickle.dump(all_models, f)

print("✓ Saved to combined_lda_models.pkl")

# Verify
with open(os.path.join(out_dir, "combined_lda_models.pkl"), 'rb') as f:
    loaded = pickle.load(f)
    print(f"Verified: {len(loaded)} models loaded")

Total models: 11
Models 1: 2, Models 2: 9
✓ Saved to combined_lda_models.pkl
Verified: 11 models loaded


In [ ]:
models = pickle.load(
    open(os.path.join(out_dir, "combined_lda_models.pkl"), "rb")
)

In [ ]:
import matplotlib.pyplot as plt

# Create figure explicitly
fig, ax = plt.subplots(figsize=(10, 6))

# Your plotting code here
model = evaluate_models(
    models,
    select_model=30,
    return_model=True
)

# Save THEN close
plt.savefig('outs/model.png', dpi=300, bbox_inches='tight')
plt.close(fig)  # Now fig is defined

In [ ]:
cistopic_obj.add_LDA_model(model)

In [ ]:
pickle.dump(
    cistopic_obj,
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "wb")
)

# Clustering and visualization

In [11]:
from pycisTopic.clust_vis import (
    find_clusters,
    run_umap,
    run_tsne,
    plot_metadata,
    plot_topic,
    cell_topic_heatmap
)

In [ ]:
find_clusters(
    cistopic_obj,
    target  = 'cell',
    k = 10,
    res = [0.6, 1.2, 3],
    prefix = 'pycisTopic_',
    scale = True,
    split_pattern = '-'
)

2026-04-14 11:55:29,425 cisTopic     INFO     Finding neighbours
Columns ['pycisTopic_leiden_10_0.6'] will be overwritten
Columns ['pycisTopic_leiden_10_1.2'] will be overwritten
Columns ['pycisTopic_leiden_10_3'] will be overwritten


In [ ]:
run_umap(
    cistopic_obj,
    target  = 'cell', scale=True)

2026-04-14 11:55:31,149 cisTopic     INFO     Running UMAP


/projectnb/paxlab/presh/env/scenicplus/lib/python3.11/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")


In [ ]:
run_tsne(
    cistopic_obj,
    target  = 'cell', scale=True)

2026-04-14 11:55:43,161 cisTopic     INFO     Running TSNE


In [ ]:
import matplotlib.pyplot as plt

plot_topic(
    cistopic_obj,
    reduction_name='UMAP',
    target='cell',
    num_columns=5
)

plt.savefig('outs/plot_topic.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Saved to outs/plot_topic.png")

✓ Saved to outs/plot_topic.png


In [ ]:
cistopic_obj.cell_data.to_csv('cell_metadata.csv', sep=',')

In [20]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import matplotlib.colors as mcolors

# Distinct palette per variable
# Clusters_tile (4 clusters) — Set2
_set2 = mpl_cm.get_cmap('Set2')
clusters_tile_colors = {cat: mcolors.to_hex(_set2(i)) for i, cat in enumerate(['C1', 'C2', 'C3', 'C4'])}

# Subclone_cluster (6 subclones) — tab10
_tab10 = mpl_cm.get_cmap('tab10')
subclone_colors = {cat: mcolors.to_hex(_tab10(i)) for i, cat in enumerate([1, 2, 3, 4, 5, 6])}

# pycisTopic_leiden (7 clusters) — Dark2
_dark2 = mpl_cm.get_cmap('Dark2')
leiden_colors = {cat: mcolors.to_hex(_dark2(i)) for i, cat in enumerate(['0', '1', '2', '3', '4', '5', '6'])}

color_dictionary = {
    'Clusters_tile': clusters_tile_colors,
    'Subclone_cluster': subclone_colors,
    'pycisTopic_leiden_10_1.2': leiden_colors,
}

# Heatmap
cell_topic_heatmap(
    cistopic_obj,
    variables = ['Clusters_tile', 'Subclone_cluster', 'pycisTopic_leiden_10_1.2'],
    color_dictionary = color_dictionary,
    scale = False,
    legend_loc_x = 1.0,
    legend_loc_y = -1.2,
    legend_dist_y = -1,
    figsize = (10, 10)
)
plt.savefig('outs/plot_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Saved heatmap to outs/plot_heatmap.png")

/scratch/ipykernel_2389787/4259916793.py:9: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  _set2 = mpl_cm.get_cmap('Set2')
/scratch/ipykernel_2389787/4259916793.py:13: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  _tab10 = mpl_cm.get_cmap('tab10')
/scratch/ipykernel_2389787/4259916793.py:17: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  _dark2 = mpl_cm.get_cmap('Dark2')


✓ Saved heatmap to outs/plot_heatmap.png


In [21]:
# UMAPs with the same color palettes
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

for ax, var in zip(axes, ['Clusters_tile', 'Subclone_cluster', 'pycisTopic_leiden_10_1.2']):
    embedding = cistopic_obj.projections['cell']['UMAP']
    meta = cistopic_obj.cell_data.loc[embedding.index, var].dropna()
    emb = embedding.loc[meta.index]
    colors = meta.map(color_dictionary[var])
    ax.scatter(emb.iloc[:, 0], emb.iloc[:, 1], c=colors, s=5, alpha=0.8)
    ax.set_title(var, fontsize=12)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    # Legend
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=8, label=str(k))
               for k, c in color_dictionary[var].items()]
    ax.legend(handles=handles, loc='best', fontsize=8, framealpha=0.7)

plt.tight_layout()
plt.savefig('outs/plot_umaps_colored.png', dpi=300, bbox_inches='tight')
plt.close()
print("✓ Saved UMAPs to outs/plot_umaps_colored.png")

✓ Saved UMAPs to outs/plot_umaps_colored.png


# Topic binarization & QC

In [60]:
import pickle
import os
import pycisTopic
pycisTopic.__version__
from pycisTopic.topic_binarization import binarize_topics

out_dir = "outs"  # or wherever you saved it

# Load the object
cistopic_obj = pickle.load(
    open(os.path.join(out_dir, "cistopic_obj.pkl"), "rb")
)

print("✓ Loaded cistopic object")
print(f"Type: {type(cistopic_obj)}")
print(f"Object info: {cistopic_obj}")

✓ Loaded cistopic object
Type: <class 'pycisTopic.cistopic_class.CistopicObject'>
Object info: CistopicObject from project Deepseq with n_cells × n_regions = 2650 × 1029858


In [61]:
# from pycisTopic.topic_binarization import binarize_topics
# region_bin_topics_top_3k = binarize_topics(
#     cistopic_obj, method='ntop', ntop = float(2000),
#     plot=True, num_columns=5
# )

In [ ]:
# Explore the object
print(dir(cistopic_obj))

In [69]:
region_bin_topics_otsu = binarize_topics(
    cistopic_obj, method='otsu',
    plot=True, num_columns=5
)



In [ ]:
binarized_cell_topic = binarize_topics(
    cistopic_obj,
    target='cell',
    method='li',
    plot=True,
    num_columns=5, nbins=100)

In [64]:
from pycisTopic.topic_qc import compute_topic_metrics, plot_topic_qc, topic_annotation
import matplotlib.pyplot as plt
from pycisTopic.utils import fig2img

In [65]:
topic_qc_metrics = compute_topic_metrics(cistopic_obj)

In [66]:
fig_dict={}
fig_dict['CoherenceVSAssignments']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Log10_Assignments', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['AssignmentsVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Log10_Assignments', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSCells_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Cells_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSRegions_in_bin']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Regions_in_binarized_topic', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSMarginal_dist']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Marginal_topic_dist', var_color='Gini_index', plot=False, return_fig=True)
fig_dict['CoherenceVSGini_index']=plot_topic_qc(topic_qc_metrics, var_x='Coherence', var_y='Gini_index', var_color='Gini_index', plot=False, return_fig=True)

KeyError: 0

In [ ]:
# Plot topic stats in one figure
fig=plt.figure(figsize=(40, 43))
i = 1
for fig_ in fig_dict.keys():
    plt.subplot(2, 3, i)
    img = fig2img(fig_dict[fig_]) #To convert figures to png to plot together, see .utils.py. This converts the figure to png.
    plt.imshow(img)
    plt.axis('off')
    i += 1
plt.subplots_adjust(wspace=0, hspace=-0.70)
plt.show()

In [32]:
topic_annot = topic_annotation(
    cistopic_obj,
    annot_var='Clusters_tile',
    binarized_cell_topic=binarized_cell_topic,
    general_topic_thr = 0.2
)

/projectnb/paxlab/presh/env/scenicplus/lib/python3.11/site-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: divide by zero encountered in scalar divide
  zstat = value / std


In [33]:
topic_annot

,Clusters_tile,Ratio_cells_in_topic,Ratio_group_in_population,is_general
Topic1,,0.295094,0.0,True
Topic2,"C2, C1, C4",0.336604,0.476226,False
Topic3,"C3, C4",0.287547,0.747925,False
Topic4,"C2, C1",0.295094,0.252075,False
Topic5,C3,0.429811,0.523774,False
Topic6,C3,0.254717,0.523774,False
Topic7,"C2, C1",0.303774,0.252075,False
Topic8,C1,0.273585,0.101887,False
Topic9,C3,0.250189,0.523774,False
Topic10,,0.250943,0.0,True


# Differentially Accessible Regions (DARs)


In [34]:
from pycisTopic.diff_features import (
    impute_accessibility,
    normalize_scores,
    find_highly_variable_features,
    find_diff_features
)
import numpy as np

In [35]:
imputed_acc_obj = impute_accessibility(
    cistopic_obj,
    selected_cells=None,
    selected_regions=None,
    scale_factor=10**6
)

2026-04-17 14:12:25,140 cisTopic     INFO     Imputing region accessibility
2026-04-17 14:12:25,141 cisTopic     INFO     Impute region accessibility for regions 0-20000
2026-04-17 14:12:25,512 cisTopic     INFO     Impute region accessibility for regions 20000-40000
2026-04-17 14:12:25,873 cisTopic     INFO     Impute region accessibility for regions 40000-60000
2026-04-17 14:12:26,246 cisTopic     INFO     Impute region accessibility for regions 60000-80000
2026-04-17 14:12:26,625 cisTopic     INFO     Impute region accessibility for regions 80000-100000
2026-04-17 14:12:26,999 cisTopic     INFO     Impute region accessibility for regions 100000-120000
2026-04-17 14:12:27,372 cisTopic     INFO     Impute region accessibility for regions 120000-140000
2026-04-17 14:12:27,736 cisTopic     INFO     Impute region accessibility for regions 140000-160000
2026-04-17 14:12:28,102 cisTopic     INFO     Impute region accessibility for regions 160000-180000
2026-04-17 14:12:28,481 cisTopic     

In [36]:
normalized_imputed_acc_obj = normalize_scores(imputed_acc_obj, scale_factor=10**4)

2026-04-17 14:12:44,299 cisTopic     INFO     Normalizing imputed data
2026-04-17 14:13:11,960 cisTopic     INFO     Done!


In [37]:
variable_regions = find_highly_variable_features(
    normalized_imputed_acc_obj,
    min_disp = 0.05,
    min_mean = 0.0125,
    max_mean = 3,
    max_disp = np.inf,
    n_bins=20,
    n_top_features=None,
    plot=True
)

2026-04-17 14:13:11,965 cisTopic     INFO     Calculating mean
2026-04-17 14:13:14,597 cisTopic     INFO     Calculating variance
2026-04-17 14:13:39,050 cisTopic     INFO     Done!


In [38]:
len(variable_regions)

60555

In [40]:
markers_dict= find_diff_features(
    cistopic_obj,
    imputed_acc_obj,
    variable='Clusters_tile',
    var_features=variable_regions,
    contrasts=None,
    adjpval_thr=0.05,
    log2fc_thr=np.log2(1.5),
    n_cpu=5,
    _temp_dir='/scratch/leuven/330/vsc33053/ray_spill',
    split_pattern = '-'
)

2026-04-17 14:14:08,074	INFO worker.py:1724 -- Started a local Ray instance.


2026-04-17 14:14:09,948 cisTopic     INFO     Subsetting data for C1 (270 of 2650)
2026-04-17 14:14:17,394 cisTopic     INFO     Computing p-value for C1
2026-04-17 14:14:27,495 cisTopic     INFO     Computing log2FC for C1
2026-04-17 14:14:29,650 cisTopic     INFO     C1 done!
2026-04-17 14:14:29,655 cisTopic     INFO     Subsetting data for C2 (398 of 2650)
2026-04-17 14:14:30,067 cisTopic     INFO     Computing p-value for C2
2026-04-17 14:14:38,909 cisTopic     INFO     Computing log2FC for C2
2026-04-17 14:14:39,118 cisTopic     INFO     C2 done!
2026-04-17 14:14:39,122 cisTopic     INFO     Subsetting data for C3 (1388 of 2650)
2026-04-17 14:14:39,535 cisTopic     INFO     Computing p-value for C3
2026-04-17 14:14:48,239 cisTopic     INFO     Computing log2FC for C3
2026-04-17 14:14:48,444 cisTopic     INFO     C3 done!
2026-04-17 14:14:48,448 cisTopic     INFO     Subsetting data for C4 (594 of 2650)
2026-04-17 14:14:48,867 cisTopic     INFO     Computing p-value for C4
2026-04-

In [41]:
print("Number of DARs found:")
print("---------------------")
for x in markers_dict:
    print(f"  {x}: {len(markers_dict[x])}")

Number of DARs found:
---------------------
  C1: 24336
  C2: 17750
  C3: 11562
  C4: 108


# Save region sets

In [42]:
os.makedirs(os.path.join(out_dir, "region_sets"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "Topics_otsu"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "Topics_top_3k"), exist_ok = True)
os.makedirs(os.path.join(out_dir, "region_sets", "DARs_cell_type"), exist_ok = True)

In [43]:
from pycisTopic.utils import region_names_to_coordinates

In [44]:
for topic in region_bin_topics_otsu:
    region_names_to_coordinates(
        region_bin_topics_otsu[topic].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "Topics_otsu", f"{topic}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [45]:
for cell_type in markers_dict:
    region_names_to_coordinates(
        markers_dict[cell_type].index
    ).sort_values(
        ["Chromosome", "Start", "End"]
    ).to_csv(
        os.path.join(out_dir, "region_sets", "DARs_cell_type", f"{cell_type}.bed"),
        sep = "\t",
        header = False, index = False
    )

In [ ]:
# for topic in region_bin_topics_top_3k:
#     region_names_to_coordinates(
#         region_bin_topics_top_3k[topic].index
#     ).sort_values(
#         ["Chromosome", "Start", "End"]
#     ).to_csv(
#         os.path.join(out_dir, "region_sets", "Topics_top_3k", f"{topic}.bed"),
#         sep = "\t",
#         header = False, index = False
#     )

# PYCISTARGET 

In [6]:
import pycistarget
pycistarget.__version__

'1.1'

In [7]:
import os
import glob
import pandas as pd
os.chdir("/projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic")

out_dir = "outs"  # change if needed
topic_dir = os.path.join(out_dir, "region_sets", "Topics_otsu")
pattern = os.path.join(topic_dir, "*.bed")  # each file: {topic}.bed

# List files
files = sorted(glob.glob(pattern))
if not files:
    raise FileNotFoundError(f"No topic BED files found at {pattern}")

# Collect regions per topic
topic_regions = {}
all_regions = set()

for f in files:
    topic = os.path.splitext(os.path.basename(f))[0]
    # Read bed: assume columns Chromosome, Start, End, ... but header=False (as saved)
    df = pd.read_csv(f, sep="\t", header=None, usecols=[0,1,2])
    df.columns = ["Chromosome", "Start", "End"]
    # Create a stable region id string
    df["region_id"] = df["Chromosome"].astype(str) + ":" + df["Start"].astype(str) + "-" + df["End"].astype(str)
    regions = df["region_id"].tolist()
    topic_regions[topic] = set(regions)
    all_regions.update(regions)

# Sort regions in natural genomic order (chr numeric, X, Y, M)
def chrom_key(chr_str):
    chr_str = chr_str.replace("chr","")
    if chr_str == "X": return (23, 0)
    if chr_str == "Y": return (24, 0)
    if chr_str in ("M","MT"): return (25, 0)
    try:
        return (int(chr_str), 0)
    except:
        return (1000, chr_str)

def region_sort_key(region_id):
    chrom, rest = region_id.split(":", 1)
    start = int(rest.split("-")[0])
    return (chrom_key(chrom), start)

sorted_regions = sorted(all_regions, key=region_sort_key)

# Build binary DataFrame: rows = regions, cols = topics
binarized_df = pd.DataFrame(0, index=sorted_regions, columns=sorted(topic_regions.keys()), dtype=int)

for topic, regs in topic_regions.items():
    binarized_df.loc[list(regs), topic] = 1

# Optionally, split region_id back to Chromosome/Start/End as multiindex or columns
regions_df = (
    pd.Series(binarized_df.index, name="region_id")
      .str.split("[:-]", expand=True)
)
regions_df.columns = ["Chromosome", "Start", "End"]
regions_df[["Start","End"]] = regions_df[["Start","End"]].astype(int)

# If you want the same structure as region_bin_topics_otsu (region index and topic columns):
binarized_topic_region = binarized_df.copy()
# set index to a BED-like MultiIndex if desired:
# binarized_topic_region.index = pd.MultiIndex.from_frame(regions_df)

# Save or inspect
print("Binarized matrix shape:", binarized_topic_region.shape)
print("Sample rows:")
print(binarized_topic_region.iloc[:5, :5])

# # If you want to restore index to (Chromosome, Start, End) tuple index:
# binarized_topic_region.index = pd.MultiIndex.from_frame(regions_df)

# Now binarized_topic_region is ready to use

Binarized matrix shape: (967360, 30)
Sample rows:
                    Topic1  Topic10  Topic11  Topic12  Topic13
chr1:802180-802680       0        0        0        1        0
chr1:804665-805165       1        0        0        0        0
chr1:807960-808460       0        0        0        0        0
chr1:811411-811911       0        0        0        0        0
chr1:812739-813239       0        0        0        0        0


In [8]:
import pyranges as pr
from pycistarget.utils import *
region_sets = {key: pr.PyRanges(region_names_to_coordinates(binarized_topic_region[key].index.tolist())) for key in binarized_topic_region.keys()}

In [9]:
from pycistarget.motif_enrichment_cistarget import *

In [10]:
db = '/projectnb/paxlab/presh/software/Rcistarget/hg38_screen_v10_clust.regions_vs_motifs.rankings.feather'
ctx_db = cisTargetDatabase(db, region_sets)
# Remove dbcorr motifs (only if using cisTarget v9)
# ctx_db.db_rankings = ctx_db.db_rankings[~ctx_db.db_rankings.index.str.contains("dbcorr")]

2026-04-30 15:45:04,805 cisTarget    INFO     Reading cisTarget database


In [11]:
for key in region_sets.keys():
    print(f'{key}: {region_sets[key].keys()}')

Topic1: ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrM', 'chrX', 'chrY']
Topic10: ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrM', 'chrX', 'chrY']
Topic11: ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrM', 'chrX', 'chrY']
Topic12: ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrM', 'chrX', 'chrY']
Topic13: ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 

In [98]:
# Run
from scenicplus.wrappers.run_pycistarget import *
import pycistarget

homo_sapiens_annot = pd.read_csv('/projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/homo_sapiens_prot_annotation.csv')


cistarget_dict = run_pycistarget(ctx_db = ctx_db, 
                               region_sets = region_sets,
                               custom_annot = homo_sapiens_annot, 
                                species = 'custom', 
                               annotation_version = 'v10nr_clust', #Use v10nr_clust for the latest motif collection
                               auc_threshold = 0.005,
                               nes_threshold = 3.0,
                               rank_threshold = 0.05,
                               annotation = ['Direct_annot', 'Orthology_annot'],
                               n_cpu = 2,
                               _temp_dir='/projectnb/paxlab/presh/tmp',
                                save_path = '/projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/pycistarget')

2026-04-23 16:41:13,803 pycisTarget_wrapper INFO     /projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/pycistarget folder already exists.
2026-04-23 16:41:13,824 pycisTarget_wrapper INFO     Saving object
2026-04-23 16:41:13,826 pycisTarget_wrapper INFO     Finished! Took 0.00038830836613972983 minutes


In [103]:
cistarget_dict = run_pycistarget(ctx_db = ctx_db, 
                               region_sets = region_sets,
                            species = 'homo_sapiens',
                               annotation_version = 'v10nr_clust', #Use v10nr_clust for the latest motif collection
                               auc_threshold = 0.005,
                               nes_threshold = 3.0,
                               rank_threshold = 0.05,
                               annotation = ['Direct_annot', 'Orthology_annot'],
                               n_cpu = 2,
                               _temp_dir='/projectnb/paxlab/presh/tmp',
                                save_path = '/projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/pycistarget')

2026-04-23 16:48:19,347 pycisTarget_wrapper INFO     /projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/pycistarget folder already exists.


BiomartException: Query ERROR: caught BioMart::Exception::Database: Could not connect to mysql database ensembl_mart_115: DBI connect('database=ensembl_mart_115;host=127.0.0.1;port=5316','ensro',...) failed: Can't connect to MySQL server on '127.0.0.1' (111) at /nfs/public/ro/ensweb/live/mart/www_115/biomart-perl/lib/BioMart/Configuration/DBLocation.pm line 98.


In [ ]:
import pickle
with open('/staging/leuven/stg_00002/lcb/cbravo/Multiomics_pipeline/analysis/10x_multiome_brain/output/atac/pycistarget/topics/topic_cistarget_dict.pkl', 'wb') as f:
  pickle.dump(cistarget_dict, f)

In [ ]:

# Load
import pickle
infile = open('/staging/leuven/stg_00002/lcb/cbravo/Multiomics_pipeline/analysis/10x_multiome_brain/output/atac/pycistarget/topics/topic_cistarget_dict.pkl', 'rb')
cistarget_dict = pickle.load(infile)
infile.close()

In [99]:
cistarget_results(cistarget_dict, name='Topic21')

NameError: name 'cistarget_results' is not defined

The Pycistarget 


# had to reinstall the scenicplus environment 
module load miniconda

mamba create -n scenicplus -c conda-forge -c bioconda \
  python=3.11 \
  pycistarget \
  polars-lts-cpu \
  --yes


mamba activate scenicplus
pip install setuptools==81.0.0 
pip install tables



cd "/projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic"
module load miniconda

mamba activate scenicplus
export POLARS_SKIP_CPU_CHECK=1

module load parallel 

# topics analysis 
## Process 4 topics in parallel - cistarget approach
find /projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/region_sets/Topics_otsu/*.bed | \
  parallel -j 4 'pycistarget cistarget \
    --cistarget_db_fname /projectnb/paxlab/presh/software/Rcistarget/hg38_screen_v10_clust.regions_vs_motifs.rankings.feather \
    --bed_fname {} \
    --output_folder /projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/pycistarget/topics_cistarget \
    --species Homo_sapiens --output_mode 'tsv' --write_html '


## Process 4 topics in parallel - dem approach 
find /projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/region_sets/Topics_otsu/*.bed | \
  parallel -j 4 'pycistarget dem \
    --dem_db_fname /projectnb/paxlab/presh/software/Rcistarget/hg38_screen_v10_clust.regions_vs_motifs.scores.feather \
    --bed_fname {} \
    --output_folder /projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/pycistarget/topics_dem \
    --species Homo_sapiens'


# DARS analysis 
## Process 4 DARS in parallel - cistarget approach
find /projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/region_sets/DARs_cell_type/*.bed | \
  parallel -j 4 'pycistarget cistarget \
    --cistarget_db_fname /projectnb/paxlab/presh/software/Rcistarget/hg38_screen_v10_clust.regions_vs_motifs.rankings.feather \
    --bed_fname {} \
    --output_folder /projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/pycistarget/dars_cistarget \
    --species Homo_sapiens --output_mode 'tsv' --write_html '


## Process 4 DARS in parallel - dem approach
find /projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/region_sets/DARs_cell_type/*.bed | \
  parallel -j 4 'pycistarget dem \
    --dem_db_fname /projectnb/paxlab/presh/software/Rcistarget/hg38_screen_v10_clust.regions_vs_motifs.scores.feather \
    --bed_fname {} \
    --output_folder /projectnb/paxlab/presh/projects/spatial_atac/Data/pycistopic/outs/pycistarget/dars_dem \
    --species Homo_sapiens'


    